# Figure S8

Maps mean reconstruction uncertainty and WTD confidence classes.


In [ ]:
from pathlib import Path
import warnings

import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap, LinearSegmentedColormap
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import pyogrio
from pyproj import Transformer

warnings.filterwarnings('ignore', category=RuntimeWarning)


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'outputs' / 'RECON_MAIN_2011_2023').exists():
            return candidate
    raise RuntimeError('Could not locate repository root.')


ROOT = find_repo_root()
RECON = ROOT / 'outputs' / 'RECON_MAIN_2011_2023'
OUT_DIR = ROOT / 'outputs' / 'figures' / 'FigS8'
OUT_DIR.mkdir(parents=True, exist_ok=True)

UNCERTAINTY_PATH = RECON / 'model_uncertainty' / 'monthly_model_uncertainty_radius_matrix.npy'
GRID_PATH = RECON / 'metadata' / 'grid_lookup.csv'
BOUNDARY_PATH = ROOT / 'data' / '0 mask MRVA' / 'outerboundary.shp'
RIVER_PATH = (
    ROOT / 'data_raw' / '11 river_network' / 'HydroRIVERS_NorthAmerica'
    / 'HydroRIVERS_v10_na_shp' / 'HydroRIVERS_v10_na.shp'
)
RIVER_ORDER_FIELD = 'ORD_CLAS'
FIGURE_PATH = OUT_DIR / 'FigS8a_mean_uncertainty_map.png'

RECON_CONFIDENCE_RADIUS_THRESHOLDS_M = (0.70, 1.50)

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Arial',
    'mathtext.it': 'Arial:italic',
    'mathtext.bf': 'Arial:bold',
    'axes.unicode_minus': False,
    'font.size': 11,
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'axes.linewidth': 0.75,
    'xtick.major.width': 0.75,
    'ytick.major.width': 0.75,
    'xtick.major.size': 3.0,
    'ytick.major.size': 3.0,
    'savefig.dpi': 900,
})


def display_path(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(ROOT))
    except ValueError:
        return str(path)


print('Reconstruction:', display_path(RECON))
print('Output:', display_path(OUT_DIR))

In [ ]:
uncertainty_radius = np.load(UNCERTAINTY_PATH, mmap_mode='r')
grid = pd.read_csv(GRID_PATH).copy()

if uncertainty_radius.ndim != 2:
    raise ValueError(
        f'Expected a two-dimensional uncertainty matrix, got {uncertainty_radius.shape}.'
    )
if uncertainty_radius.shape[1] != len(grid):
    raise ValueError(
        f'Uncertainty columns ({uncertainty_radius.shape[1]}) do not match '
        f'grid rows ({len(grid)}).'
    )

grid['mean_pi75_radius_m'] = np.nanmean(uncertainty_radius, axis=0)

row_min = int(grid['row'].min())
row_max = int(grid['row'].max())
col_min = int(grid['col'].min())
col_max = int(grid['col'].max())
n_rows = row_max - row_min + 1
n_cols = col_max - col_min + 1
active_row = (grid['row'] - row_min).to_numpy(dtype=int)
active_col = (grid['col'] - col_min).to_numpy(dtype=int)

mean_uncertainty_grid = np.full((n_rows, n_cols), np.nan, dtype=float)
mean_uncertainty_grid[active_row, active_col] = grid[
    'mean_pi75_radius_m'
].to_numpy(dtype=float)

high_threshold, medium_threshold = RECON_CONFIDENCE_RADIUS_THRESHOLDS_M
mean_radius = grid['mean_pi75_radius_m'].to_numpy(dtype=float)
finite_radius = np.isfinite(mean_radius)
confidence_category = np.full(len(grid), np.nan, dtype=float)
confidence_category[finite_radius] = np.select(
    [
        mean_radius[finite_radius] <= high_threshold,
        mean_radius[finite_radius] <= medium_threshold,
    ],
    [2.0, 1.0],
    default=0.0,
)
confidence_category_grid = np.full((n_rows, n_cols), np.nan, dtype=float)
confidence_category_grid[active_row, active_col] = confidence_category

summary = pd.Series({
    'Mean radius (m)': np.nanmean(mean_radius),
    'High-confidence fraction': np.nanmean(confidence_category == 2.0),
    'Medium-confidence fraction': np.nanmean(confidence_category == 1.0),
    'Low-confidence fraction': np.nanmean(confidence_category == 0.0),
})
summary.round(3)

In [ ]:
boundary = gpd.read_file(BOUNDARY_PATH)
if boundary.crs is None:
    boundary = boundary.set_crs('EPSG:5070')
boundary = boundary.to_crs('EPSG:5070')

river_bbox = tuple(float(value) for value in boundary.to_crs('EPSG:4326').total_bounds)
rivers = pyogrio.read_dataframe(
    RIVER_PATH,
    bbox=river_bbox,
    where=f'{RIVER_ORDER_FIELD} = 1',
    columns=[RIVER_ORDER_FIELD],
    use_arrow=True,
)
rivers = rivers.to_crs(boundary.crs)
rivers = gpd.clip(rivers, boundary)
rivers = rivers[rivers.geometry.notna() & (~rivers.geometry.is_empty)].copy()

x_by_col = grid.groupby('col')['x'].first().sort_index().to_numpy(dtype=float)
y_by_row = grid.groupby('row')['y'].first().sort_index().to_numpy(dtype=float)
dx = float(np.nanmedian(np.diff(x_by_col)))
dy = float(np.nanmedian(np.diff(y_by_row)))
x_edges = np.r_[x_by_col[0] - 0.5 * dx, x_by_col + 0.5 * dx]
y_edges = np.r_[y_by_row[0] - 0.5 * dy, y_by_row + 0.5 * dy]
x_edge_grid, y_edge_grid = np.meshgrid(x_edges, y_edges)

transformer = Transformer.from_crs('EPSG:5070', 'EPSG:4326', always_xy=True)
lon_edges, lat_edges = transformer.transform(x_edge_grid, y_edge_grid)
lon_edges = np.asarray(lon_edges)
lat_edges = np.asarray(lat_edges)
map_extent = (
    float(np.nanmin(lon_edges)),
    float(np.nanmax(lon_edges)),
    float(np.nanmin(lat_edges)),
    float(np.nanmax(lat_edges)),
)
mean_latitude = 0.5 * (map_extent[2] + map_extent[3])
boundary_lonlat = boundary.to_crs('EPSG:4326')
rivers_lonlat = rivers.to_crs('EPSG:4326')


def plot_geographic_context(axis):
    if not rivers_lonlat.empty:
        rivers_lonlat.plot(
            ax=axis,
            color='#72B9D3',
            linewidth=0.68,
            alpha=0.98,
            zorder=3,
        )
    boundary_lonlat.boundary.plot(
        ax=axis,
        color='#1f1f1f',
        linewidth=0.45,
        zorder=4,
    )
    axis.set_xlim(map_extent[0], map_extent[1])
    axis.set_ylim(map_extent[2], map_extent[3])
    axis.set_xticks([])
    axis.set_yticks([])
    axis.tick_params(
        axis='both',
        which='both',
        bottom=False,
        left=False,
        labelbottom=False,
        labelleft=False,
    )
    axis.set_aspect(1.0 / np.cos(np.deg2rad(mean_latitude)))
    for spine in axis.spines.values():
        spine.set_visible(False)

In [ ]:
uncertainty_cmap = LinearSegmentedColormap.from_list(
    'pi75_uncertainty',
    ['#f7fbff', '#deebf7', '#9ecae1', '#4292c6', '#08519c', '#08306b'],
)
confidence_cmap = ListedColormap([
    '#d8e3e6',
    '#83b6be',
    '#2e6f73',
])
confidence_cmap.set_bad('#FFFFFF')
confidence_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], 3)

figure = plt.figure(figsize=(7.8, 5.5), constrained_layout=True)
grid_spec = figure.add_gridspec(1, 2, width_ratios=[1.15, 1.0])
uncertainty_axis = figure.add_subplot(grid_spec[0, 0])
confidence_axis = figure.add_subplot(grid_spec[0, 1])

uncertainty_image = uncertainty_axis.pcolormesh(
    lon_edges,
    lat_edges,
    mean_uncertainty_grid,
    cmap=uncertainty_cmap,
    vmin=0.0,
    vmax=4.0,
    shading='flat',
    rasterized=True,
)
confidence_axis.pcolormesh(
    lon_edges,
    lat_edges,
    confidence_category_grid,
    cmap=confidence_cmap,
    norm=confidence_norm,
    shading='flat',
    rasterized=True,
)

for axis in [uncertainty_axis, confidence_axis]:
    plot_geographic_context(axis)

uncertainty_axis.set_title(
    'Mean reconstruction uncertainty',
    loc='left',
    fontweight='bold',
    fontsize=9,
    pad=4,
)
uncertainty_colorbar = figure.colorbar(
    uncertainty_image,
    ax=uncertainty_axis,
    fraction=0.045,
    pad=0.025,
    ticks=[0, 1, 2, 3, 4],
    extend='max',
)
uncertainty_colorbar.set_label('Mean uncertainty radius (m)', fontsize=8.5)
uncertainty_colorbar.ax.tick_params(labelsize=7.5)

confidence_axis.set_title(
    'WTD reconstruction confidence',
    loc='left',
    fontweight='bold',
    fontsize=9,
    pad=4,
)
confidence_handles = [
    Patch(facecolor='#2e6f73', edgecolor='none', label='High'),
    Patch(facecolor='#83b6be', edgecolor='none', label='Medium'),
    Patch(facecolor='#d8e3e6', edgecolor='none', label='Low'),
]
confidence_axis.legend(
    handles=confidence_handles,
    frameon=False,
    loc='upper left',
    bbox_to_anchor=(0.0, -0.02),
    borderaxespad=0.0,
    fontsize=7.5,
    handlelength=1.0,
    handletextpad=0.4,
)

figure.savefig(FIGURE_PATH, bbox_inches='tight', dpi=900, facecolor='white')
plt.close(figure)

print('Saved:')
print('  ' + display_path(FIGURE_PATH))
print(f'  High-confidence radius threshold: <= {high_threshold:.2f} m')
print(f'  Medium-confidence radius threshold: <= {medium_threshold:.2f} m')